# Entanglement Optimization: JAX TPU Benchmark

This notebook benchmarks the core geometric kernels (rod-rod distance and linking number) on **Google Colab TPUs**.

## Instructions
1.  Go to **Runtime -> Change runtime type**.
2.  Select **Hardware accelerator: TPU v2** (or latest available).
3.  Run all cells.

In [ ]:
import jax
import jax.numpy as jnp
from jax import jit, vmap, grad, lax
import timeit
import matplotlib.pyplot as plt
import numpy as np

print("JAX Version:", jax.__version__)
try:
    print("Available Devices:", jax.devices())
except RuntimeError:
    print("No TPU/GPU found. Please enable it in Runtime settings.")

## Core Physics Definitions
We include the JAX implementations of `dist_lin_seg` and `compute_linking_number` here directly to make this notebook self-contained.

In [ ]:
def fixbound(num):
    """Ensure the number is within the bounds [0, 1]."""
    return jnp.clip(num, 0.0, 1.0)

@jit
def dist_lin_seg(point1s, point1e, point2s, point2e):
    """Calculate the shortest distance between two line segments using JAX with cond."""
    d1 = point1e - point1s
    d2 = point2e - point2s
    d12 = point2s - point1s

    D1 = jnp.dot(d1, d1)
    D2 = jnp.dot(d2, d2)
    S1 = jnp.dot(d1, d12)
    S2 = jnp.dot(d2, d12)
    R = jnp.dot(d1, d2)

    den = D1 * D2 - R**2
    
    def case1():
        (t,u) = lax.cond( D1 != 0. , 
                    lambda _: (fixbound(S1/D1), 0.),
                    lambda _: lax.cond(D2 != 0.,
                             lambda _: (0., fixbound(-S2/D2)),
                             lambda _: (0., 0.),
                             None),
                    None)        
        return (t,u)
    
    def case2_1():
        t = 0.
        u = -S2/D2
        uf = fixbound(u)
        
        (t,u) = lax.cond(uf != u, 
                    lambda _: (fixbound((uf * R + S1) / D1), uf),
                    lambda _: (t, u),
                    None)
        return (t,u)
    
    def case2_2():
        t = fixbound((S1 * D2 - S2 * R) / den)
        u = (t * R - S2) / D2
        uf = fixbound(u)
        
        (t,u) = lax.cond(uf != u, 
                    lambda _: (fixbound((uf * R + S1) / D1), uf),
                    lambda _: (t, u),
                    None)
        return (t,u)        
    
    def case2():
        (t,u) = lax.cond( den == 0. , 
                    lambda _: case2_1(),                    
                    lambda _: case2_2(),
                    None)        
        return (t,u)
    
    (t,u) = lax.cond( (D1 == 0.) & (D2 == 0.),
                        lambda _: case1(),
                        lambda _: case2(),
                        None)
    
    dist = jnp.linalg.norm(d1 * t - d2 * u - d12)
    return dist

@jit
def compute_linking_number_cartesian(p_i, p_ii, p_j, p_jj):
    r_ij = p_i - p_j
    r_ijj = p_i - p_jj
    r_iij = p_ii - p_j
    r_iijj = p_ii - p_jj

    tol = 1e-6
    n1 = jnp.cross(r_ij, r_ijj)
    n1 = n1/(jnp.linalg.norm(n1)+tol)
    n2 = jnp.cross(r_ijj, r_iijj)
    n2 = n2/(jnp.linalg.norm(n2)+tol)
    n3 = jnp.cross(r_iijj, r_iij)
    n3 = n3/(jnp.linalg.norm(n3)+tol)
    n4 = jnp.cross(r_iij, r_ij)
    n4 = n4/(jnp.linalg.norm(n4)+tol)
    
    tol_clip = 0.
    return -1.0/(4.0*jnp.pi)*jnp.abs(jnp.arcsin(jnp.clip(jnp.dot(n1,n2),-1.+tol_clip,1.-tol_clip))
                                   + jnp.arcsin(jnp.clip(jnp.dot(n2,n3),-1.+tol_clip,1.-tol_clip))
                                   + jnp.arcsin(jnp.clip(jnp.dot(n3,n4),-1.+tol_clip,1.-tol_clip))
                                   + jnp.arcsin(jnp.clip(jnp.dot(n4,n1),-1.+tol_clip,1.-tol_clip)))


## Workload Definition
We compute the gradient of the total potential (sum of distances + sum of linking numbers) for $N$ pairs.
This is fully vectorized using `vmap`.

In [ ]:
@jit
def jax_workload(p1s, p1e, p2s, p2e):
    # Vectorized computation 
    v_dist = vmap(dist_lin_seg)
    v_lk = vmap(compute_linking_number_cartesian)
    
    dists = v_dist(p1s, p1e, p2s, p2e)
    lks = v_lk(p1s, p1e, p2s, p2e)
    
    return jnp.sum(dists) + jnp.sum(lks)

# JIT-compile the gradient function
grad_fn = jit(grad(jax_workload, argnums=(0,1,2,3)))

## Run Benchmark (Single Step)

In [ ]:
def run_benchmark(n_pairs):
    key = jax.random.PRNGKey(0)
    
    # Initialize random rod endpoints
    p1s = jax.random.uniform(key, (n_pairs, 3))
    p1e = jax.random.uniform(key, (n_pairs, 3))
    p2s = jax.random.uniform(key, (n_pairs, 3))
    p2e = jax.random.uniform(key, (n_pairs, 3))
    
    # Warmup
    print(f"Warmup for N={n_pairs}...")
    grads = grad_fn(p1s, p1e, p2s, p2e)
    grads[0].block_until_ready()
    
    # Benchmark
    print("Running timing...")
    start_time = timeit.default_timer()
    
    grads = grad_fn(p1s, p1e, p2s, p2e)
    grads[0].block_until_ready()
    
    end_time = timeit.default_timer()
    total_time = end_time - start_time
    
    return total_time

# Test sizes: 10k, 100k, 1M, 5M (to really test TPU)
N_LIST = [10_000, 100_000, 1_000_000, 5_000_000]
times = []

print(f"{'N Pairs':<15} | {'Total Time (s)':<15} | {'us/pair':<15}")
print("-"*50)

for n in N_LIST:
    try:
        t = run_benchmark(n)
        us_per_pair = (t * 1e6) / n
        times.append(us_per_pair)
        print(f"{n:<15} | {t:.4f}          | {us_per_pair:.4f}")
    except Exception as e:
        print(f"{n:<15} | ERROR: {e}")
        times.append(0)


## Protocol Benchmark (Full Optimization Loop)
We now benchmark the full `standard_protocol` equivalent: 300 iterations of FIRE optimization.

In [ ]:
def optimize_fire(p1s, p1e, p2s, p2e, Nmax=300):
   # Simple FIRE implementation for TPU benchmark
   # q packed as (p1s, p1e, p2s, p2e) - simplified for demo
   
   dt = 0.01
   dtmax = 10 * dt
   dtmin = 0.02 * dt
   alpha0 = 0.1
   finc = 1.1
   fdec = 0.5
   fa = 0.99
   
   N = p1s.shape[0]
   vals = (p1s, p1e, p2s, p2e)
   vels = (jnp.zeros_like(p1s), jnp.zeros_like(p1e), jnp.zeros_like(p2s), jnp.zeros_like(p2e))
   
   # state: vals, vels, alpha, dt, Npos
   init_state = (vals, vels, alpha0, dt, 0)
   
   def body_fun(carry, i):
       (curr_vals, curr_vels, alpha, dt, Npos) = carry
       
       # Gradients
       grads = grad_fn(*curr_vals)
       F = tuple(-g for g in grads)
       
       # Power
       P = sum(jnp.sum(f * v) for f, v in zip(F, curr_vels))
       
       # Update hyperparameters
       dt = jax.lax.cond(P > 0, 
                         lambda _: jnp.minimum(dt * finc, dtmax),
                         lambda _: jnp.maximum(dt * fdec, dtmin),
                         None)
       alpha = jax.lax.cond(P > 0, lambda _: alpha * fa, lambda _: alpha0, None)
       Npos = jax.lax.cond(P > 0, lambda _: Npos + 1, lambda _: 0, None)
       
       # FIRE Velocity Update
       norm_V = jnp.sqrt(sum(jnp.sum(v**2) for v in curr_vels))
       norm_F = jnp.sqrt(sum(jnp.sum(f**2) for f in F))
       
       def update_v(v, f):
            return (1 - alpha) * v + alpha * f * norm_V / (norm_F + 1e-8)
            
       from jax.tree_util import tree_map
       curr_vels = tree_map(update_v, curr_vels, F)

       # Integration (Semi-Implicit Euler)
       curr_vels = tuple(v + dt * f for v, f in zip(curr_vels, F))
       curr_vals = tuple(x + dt * v for x, v in zip(curr_vals, curr_vels))
       
       return (curr_vals, curr_vels, alpha, dt, Npos), None

   final_state, _ = lax.scan(body_fun, init_state, jnp.arange(Nmax))
   return final_state[0]

def run_protocol_benchmark(n_pairs, n_iter=300):
   key = jax.random.PRNGKey(0)
   p1s = jax.random.uniform(key, (n_pairs, 3))
   p1e = jax.random.uniform(key, (n_pairs, 3))
   p2s = jax.random.uniform(key, (n_pairs, 3))
   p2e = jax.random.uniform(key, (n_pairs, 3))
   
   print(f"Benchmarking Protocol (N={n_pairs}, Iter={n_iter})... (Warmup)")
   
   # JIT compile
   optimizer_jit = jit(optimize_fire, static_argnums=(4,))
   
   # Warmup
   res = optimizer_jit(p1s, p1e, p2s, p2e, 10)
   res[0].block_until_ready()
   
   print("Running timing...")
   start = timeit.default_timer()
   res = optimizer_jit(p1s, p1e, p2s, p2e, n_iter)
   res[0].block_until_ready()
   end = timeit.default_timer()
   
   print(f"Total Time: {end-start:.4f} s")
   print(f"Time per Iter: {(end-start)/n_iter:.6f} s")
   return end-start

print("\n--- Full Protocol Benchmark ---")
for n in [1000, 10000, 100000]:
    try:
        run_protocol_benchmark(n)
    except Exception as e:
        print(e)
